# Notebook 02 - Reconstruction and Residuals

**Ramdas Chaugale - 40494109**  
**ECS8056 Themed Research Project**

Fine-tuning the OCT-F foundation model from human retina onto animal retina, and
measuring what the residual between an image and its reconstruction actually carries.
Training happened on the HPC in `gpu_01`; this notebook loads that checkpoint and
analyses it on CPU.

## 1. Setup

Loads the fine-tuned checkpoint written by `gpu_01` and the residual cache derived from it. No
training occurs here; the foundation model is 23.5M parameters and was fine-tuned on an A100.
The OCT-F source is third-party and sits outside the repository, so `sys.path` is extended to
reach it.

In [ ]:
import os
import sys
import pickle
import time

import numpy as np
import matplotlib.pyplot as plt
import cv2
import torch
from PIL import Image

PROJECT_ROOT = os.path.dirname(os.path.abspath(os.getcwd()))

DATA_DIR        = os.path.join(PROJECT_ROOT, 'data')
ENDPOINT_FOLDER = os.path.join(DATA_DIR, 'endpoint')
QC_FOLDER       = os.path.join(DATA_DIR, 'baseline_qc')
QC_FOLDER_B   = os.path.join(DATA_DIR, 'baseline_qc_cohort_b')
ANNO_FOLDER     = os.path.join(DATA_DIR, 'annotated',
                               'endpoint_annotated')
MASK_FOLDER     = os.path.join(DATA_DIR, 'masks')

OCTF_CODE      = os.path.join(PROJECT_ROOT, 'reconstruction',
                              'OCT-F_Reconstruction', 'code')
FINETUNE_CKPT  = os.path.join(PROJECT_ROOT, 'reconstruction',
                              'fine_tuned_reconstruction.pth')
RESIDUAL_CACHE = os.path.join(PROJECT_ROOT, 'reconstruction',
                              'residual_cache.pkl')

IMAGES_DIR = os.path.join(PROJECT_ROOT, 'outputs', 'images')
os.makedirs(IMAGES_DIR, exist_ok=True)

# the OCT-F source was supplied by its authors and sits outside this project
sys.path.insert(0, OCTF_CODE)

print("Project root:", PROJECT_ROOT)

In [ ]:
SEED = 40494109
np.random.seed(SEED)
torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# one plotting style for the whole notebook
plt.rcParams.update({
    'figure.dpi': 100, 'savefig.dpi': 150, 'font.size': 11,
    'axes.titlesize': 12, 'axes.titleweight': 'bold', 'axes.labelsize': 11,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.25, 'legend.frameon': False,
    'figure.titlesize': 14, 'figure.titleweight': 'bold',
})

TRAIN     = '#2F6FA8'
VALID     = '#C0392B'
LESION    = '#2E9E5B'
HIGHLIGHT = '#C0392B'

print(f"Seed: {SEED}")
print(f"Device: {device}")

In [ ]:
from architectureV2Res_new import ArchitectureV2

# weights_only=False because the checkpoint stores history and config,
# not just tensors
ckpt = torch.load(FINETUNE_CKPT, map_location=device, weights_only=False)
model = ArchitectureV2().to(device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

parameter_count = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {parameter_count:,} parameters")

In [ ]:
# residuals were computed once in gpu_01 and cached, so every model later
# sees identical inputs and no result depends on recomputing them
with open(RESIDUAL_CACHE, 'rb') as f:
    residual_cache = pickle.load(f)

anno_files = sorted([f for f in os.listdir(ANNO_FOLDER) if f.endswith('.tif')])
cached_shapes = set()
for name in residual_cache:
    cached_shapes.add(residual_cache[name].shape)

print(f"Residuals cached: {len(residual_cache)}")
print(f"Cached at shape:  {cached_shapes}")
print(f"Annotated endpoint images: {len(anno_files)}")

Execution is CPU-only: the checkpoint and the cached residuals are the sole inputs, so nothing
in this notebook requires a GPU to reproduce. Residuals are cached at the native 496x768 geometry
for all 74 annotated images. Caching once is a design decision every two-channel model in
notebooks 03 to 05 reads the same file, so no observed difference between them can be an artefact
of recomputation.

## 2. Fine-tuning configuration

Configuration is read from the checkpoint rather than restated, so the notebook cannot drift from
the run that produced the weights. The QC train/held-out split is also stored in the checkpoint,
which permits the zero-overlap assertion below.

In [ ]:
cfg = ckpt['config']
settings = [
    ('Base model', 'OCT-F, pretrained on NICOLA human retinal OCT'),
    ('Fine-tuned on', f"{cfg['qc_train']} baseline QC scans"),
    ('Held out', f"{cfg['qc_val']} QC images, used only for SSIM"),
    ('Input size', f"{cfg['img_size'][0]} x {cfg['img_size'][1]}"),
    ('Batch size', cfg['batch_size']),
    ('Optimiser', f"{cfg['optimizer']}, lr {cfg['lr']}, weight decay {cfg['weight_decay']}"),
    ('Scheduler', cfg['scheduler']),
    ('Loss', f"{cfg['loss_function']} (0.5 MSE + 0.5 (1 - SSIM))"),
    ('Max epochs', cfg['max_epochs']),
    ('Early stopping', f"patience {cfg['patience']}, fired at epoch {cfg['early_stopped_at']}"),
    ('Best epoch', ckpt['epoch']),
    ('Seed', cfg['seed']),
    ('Parameters', f"{cfg['params']:,}"),
]

print(f"{'Setting':<18} Value")
print("=" * 74)
for name, value in settings:
    print(f"{name:<18} {value}")

In [ ]:
# the exact split is stored in the checkpoint, so the held-out set can be
# verified rather than taken on trust
train_files = ckpt['qc_train_files']
val_files_qc = ckpt['qc_val_files']
overlap = set(train_files) & set(val_files_qc)

print(f"QC images used for fine-tuning: {len(train_files)}")
print(f"QC images held out:             {len(val_files_qc)}")
print(f"Overlap between the two sets:   {len(overlap)}")

OCT-F was pretrained on human retinal OCT; fine-tuning adapts it to murine retina using **QC
images only**. Restricting to QC baseline scans is the design decision the residual channel depends
on: a model fitted to lesion appearance would reconstruct lesions accurately and eliminate the
signal. 19 QC images were withheld, overlap **0**, which is what makes the SSIM figure in Section
4 a generalisation measure rather than a training statistic.

## 3. Convergence

Per-epoch loss, SSIM and learning rate, read from the history stored in the checkpoint. The
quantity of interest is the train-validation SSIM separation, which distinguishes adaptation from
memorisation on a 74-image fine-tuning set.

In [ ]:
h = ckpt['history']
epochs = np.arange(1, len(h['train_loss']) + 1)
best = ckpt['epoch']

fig, axes = plt.subplots(1, 3, figsize=(16, 4.0))

# train and validation on the same axes: if they diverge, the model is
# fitting the training images rather than learning the task
axes[0].plot(epochs, h['train_loss'], label='train', color=TRAIN)
axes[0].plot(epochs, h['val_loss'], label='validation', color=VALID)
axes[0].axvline(best, color='0.4', linestyle='--', label=f"best epoch {best}")
axes[0].set_xlabel('epoch')
axes[0].set_ylabel('loss')
axes[0].set_title('Reconstruction loss')
axes[0].legend()

axes[1].plot(epochs, h['train_ssim'], label='train', color=TRAIN)
axes[1].plot(epochs, h['val_ssim'], label='validation', color=VALID)
axes[1].axvline(best, color='0.4', linestyle='--')
axes[1].set_xlabel('epoch')
axes[1].set_ylabel('SSIM')
axes[1].set_title('SSIM, higher is better')
axes[1].legend()

# log scale, because cosine annealing spans several orders of magnitude
axes[2].plot(epochs, h['lr'], color=LESION)
axes[2].set_xlabel('epoch')
axes[2].set_ylabel('learning rate')
axes[2].set_title('Cosine annealing schedule')
axes[2].set_yscale('log')

fig.suptitle(f"OCT-F fine-tuning: {len(epochs)} epochs, best at epoch {best}", y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(IMAGES_DIR, '02_finetuning_curves.png'), bbox_inches='tight')
plt.show()

**How to read it.** Blue is performance on the images the model learned from; red is performance on
images it never saw. The dashed vertical line marks the version of the model that was kept.

The healthy pattern is the two colours staying together. If the model were simply memorising its
training images, blue would keep improving while red stopped or reversed. It does not. Red is
bumpier than blue only because it is measured on 19 images rather than 74.

The right panel is the step size, which shrinks on a fixed schedule that is why the other two
curves flatten towards the end.

In [ ]:
gap = h['train_ssim'][-1] - h['val_ssim'][-1]

print(f"Epochs run:          {len(epochs)}")
print(f"Best epoch:          {best}")
print(f"Early stopping at:   {cfg['early_stopped_at']}")
print(f"Train loss:          {h['train_loss'][0]:.4f} -> {h['train_loss'][-1]:.4f}")
print(f"Validation loss:     {h['val_loss'][0]:.4f} -> {h['val_loss'][-1]:.4f} "
      f"(best {min(h['val_loss']):.4f})")
print(f"Train/val SSIM gap at the final epoch: {gap:.4f}")

Early stopping fired at epoch 41 with the retained model at epoch 31, so the run terminated on
its own criterion rather than an epoch budget. Train-validation SSIM separation at the final epoch
is **0.0218**, small relative to the 0.4063 improvement, indicating the adaptation generalised
rather than fitting the 74 fine-tuning images.

## 4. Held-out reconstruction quality

SSIM on the 19 withheld QC images, before and after fine-tuning. Per-image values are reported
alongside the mean so that the improvement cannot be attributed to a small number of images, and
the held-out set is then broken down by study so the figure is not read as describing one cohort
only.

In [ ]:
improvement = ckpt['held_out_ssim'] - ckpt['baseline_ssim']

print(f"{'Stage':<22} {'SSIM':>8} {'Std':>8}")
print("=" * 40)
print(f"{'Before fine-tuning':<22} {ckpt['baseline_ssim']:>8.4f} "
      f"{ckpt['baseline_ssim_std']:>8.4f}")
print(f"{'After fine-tuning':<22} {ckpt['held_out_ssim']:>8.4f} "
      f"{ckpt['held_out_ssim_std']:>8.4f}")
print(f"\nImprovement: {improvement:+.4f}")

In [ ]:
# per-image SSIM, so a single good image cannot be carrying the average
per_image = np.array(ckpt['per_image_ssim'])
order = np.argsort(-per_image)

print(f"Per-image SSIM on the {len(val_files_qc)} held-out QC images")
print(f"{'File':<24} {'SSIM':>8}")
print("-" * 34)
for index in order:
    print(f"{val_files_qc[index][:-4]:<24} {per_image[index]:>8.4f}")

print(f"\nmin {per_image.min():.4f}   mean {per_image.mean():.4f}   "
      f"max {per_image.max():.4f}")

In [ ]:
# the held-out set has to span both studies, or the SSIM figure would describe
# only one cohort; the two cohorts use different subject id prefixes
thb = 0
xit = 0
for name in val_files_qc:
    if name.startswith('152'):
        thb += 1
    else:
        xit += 1

print(f"Held-out composition: {thb} from cohort A, {xit} from cohort B")

Held-out SSIM **0.2427 → 0.6491**, improvement **+0.4063**, on images that contributed no
weight update. Per-image range 0.4051 to 0.7696 across both studies, so the gain is distributed
rather than driven by outliers.

## 5. Residual construction

`reconstruct()` implements the forward pass at the model's fixed 496x768 input and transposes the
decoder output, which returns 768x496. The residual is the absolute difference. The encoder
concatenates a random tensor per forward pass, so the seed is reset before the examples to keep
them reproducible.

In [ ]:
# OCT-F only accepts 496x768, and its decoder returns the image transposed,
# so the output is transposed back before differencing
def reconstruct(image_gray):
    """Return (original, reconstruction, residual) at 496 x 768."""
    resized = cv2.resize(image_gray, (768, 496))
    x = torch.from_numpy(resized).float().unsqueeze(0).unsqueeze(0).to(device) / 255.0
    with torch.no_grad():
        recon, _ = model(x)
    original = x.squeeze().cpu().numpy()
    reconstruction = recon.squeeze().cpu().numpy().T
    residual = np.abs(original - reconstruction)
    return original, reconstruction, residual


# the encoder concatenates a random tensor on every forward pass, so the
# seed is reset here to keep these examples reproducible
torch.manual_seed(SEED)

examples = [(QC_FOLDER, 'a baseline QC scan', 'QC baseline'),
            (ENDPOINT_FOLDER, 'an endpoint scan', 'Endpoint, lesion present')]

fig, axes = plt.subplots(2, 3, figsize=(16, 6.2))
for row in range(2):
    folder, name, label = examples[row]
    gray = np.array(Image.open(os.path.join(folder, name)))[:, :, 0]
    original, reconstruction, residual = reconstruct(gray)

    axes[row][0].imshow(original, cmap='gray', vmin=0, vmax=1)
    axes[row][0].set_title(f"{label}\n{name[:-4]}", fontsize=10)
    axes[row][1].imshow(reconstruction, cmap='gray', vmin=0, vmax=1)
    axes[row][1].set_title('Reconstruction', fontsize=10)

    # fixed vmax so the two rows share a colour scale and can be compared
    picture = axes[row][2].imshow(residual, cmap='hot', vmin=0, vmax=0.5)
    axes[row][2].set_title(f"Residual  |image − reconstruction|\n"
                           f"mean {residual.mean():.4f}", fontsize=10)
    fig.colorbar(picture, ax=axes[row][2], fraction=0.03)

    for col in range(3):
        axes[row][col].axis('off')

fig.suptitle('The reconstruction smooths towards the QC baseline appearance; '
             'the residual marks where that failed', y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(IMAGES_DIR, '02_residual_examples.png'), bbox_inches='tight')
plt.show()

**How to read it.** Top row is the QC baseline scan, bottom row the endpoint scan of the same eye. Left is
the real image, middle is the model's attempt to redraw it, right is the difference between the two
on a heat scale where black means no error and bright means large error.

Two honest things to notice. The difference map is **never empty** the model is imperfect
everywhere, not only at the lesion. And its **brightest parts are the left and right edges of the
frame**, which is an artefact of padding rather than anything anatomical.

Reconstruction error is low on the QC row and elevated on the endpoint row, concentrated at the
site of tissue disruption. This spatial localisation is the property the second input channel
exploits. The residual arrays shown here are the same ones cached in `residual_cache.pkl` and
consumed by the two-channel models in notebooks 03 to 05.

## 6. Whole-image SSIM as a detector

Reconstruction-based anomaly detection assumes diseased tissue reconstructs worse than healthy
tissue. Testing that assumption directly determines whether a scalar reconstruction score is
sufficient, or whether spatial resolution of the error is required. SSIM is computed with the same
`pytorch_msssim` implementation used on the HPC, so values are comparable with the checkpoint.

In [ ]:
# same SSIM implementation used on Kelvin, so these values are directly
# comparable with the per-image figures stored in the checkpoint
from pytorch_msssim import ssim as ssim_fn


def image_ssim(folder, name):
    """Whole-image SSIM between a scan and its reconstruction."""
    gray = np.array(Image.open(os.path.join(folder, name)))[:, :, 0]
    resized = cv2.resize(gray, (768, 496))
    x = torch.from_numpy(resized).float().unsqueeze(0).unsqueeze(0).to(device) / 255.0
    with torch.no_grad():
        recon, _ = model(x)
    recon = recon.permute(0, 1, 3, 2)
    return float(ssim_fn(recon, x, data_range=1.0, size_average=True))


torch.manual_seed(SEED)
start = time.time()

qc_scores = []
for name in val_files_qc:
    folder = QC_FOLDER if name.startswith('152') else QC_FOLDER_B
    qc_scores.append(image_ssim(folder, name))

ep_scores = []
for name in anno_files:
    ep_scores.append(image_ssim(ENDPOINT_FOLDER, name))

qc_scores = np.array(qc_scores)
ep_scores = np.array(ep_scores)

print(f"Computed {len(qc_scores) + len(ep_scores)} reconstructions "
      f"in {time.time() - start:.0f}s")

In [ ]:
difference = qc_scores.mean() - ep_scores.mean()
relative = difference / qc_scores.mean() * 100

print(f"{'Image type':<28} {'n':>4} {'mean':>8} {'std':>8} {'min':>8} {'max':>8}")
print("=" * 68)
print(f"{'QC (held out, baseline)':<28} {len(qc_scores):>4} {qc_scores.mean():>8.4f} "
      f"{qc_scores.std():>8.4f} {qc_scores.min():>8.4f} {qc_scores.max():>8.4f}")
print(f"{'Endpoint (lesion present)':<28} {len(ep_scores):>4} {ep_scores.mean():>8.4f} "
      f"{ep_scores.std():>8.4f} {ep_scores.min():>8.4f} {ep_scores.max():>8.4f}")
print(f"\nDifference: {difference:+.4f}  ({relative:+.1f}% relative)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.0))

# the box spans the middle half of the images, the line inside is the median
box = axes[0].boxplot([qc_scores, ep_scores],
                      tick_labels=[f"QC\n(n={len(qc_scores)})",
                                   f"Endpoint\n(n={len(ep_scores)})"],
                      patch_artist=True, widths=0.5)
for patch, colour in zip(box['boxes'], [TRAIN, VALID]):
    patch.set_facecolor(colour)
    patch.set_alpha(0.65)
axes[0].set_ylabel('SSIM')
axes[0].set_title('The two distributions sit on top of each other')

axes[1].hist(qc_scores, bins=12, alpha=0.65, label='QC baseline', color=TRAIN)
axes[1].hist(ep_scores, bins=12, alpha=0.65, label='Endpoint (lesion)', color=VALID)
axes[1].set_xlabel('SSIM')
axes[1].set_ylabel('images')
axes[1].set_title(f"Difference {difference:+.4f}: not a detector")
axes[1].legend()

fig.suptitle('Whole-image SSIM does not separate QC baseline from endpoint scans', y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(IMAGES_DIR, '02_ssim_qc_vs_endpoint.png'), bbox_inches='tight')
plt.show()

**How to read it.** This figure exists to show that **nothing separates**, which is an unusual thing
for a figure to do.

The left panel summarises each group as a box covering its middle half. The right panel is the same
data as two overlaid histograms, where the dark region is where they sit on top of each other. If QC baseline and endpoint scans reconstructed differently, the two shapes would sit apart. They do not
and the diseased group is very slightly *higher*, the opposite of what was expected.

**Negative result.** Held-out QC scans reconstruct at 0.6490 against 0.6539 for endpoint scans a difference of **−0.0050**, in the opposite direction to the hypothesis. A scalar score over
the full frame cannot resolve a lesion occupying 2.17% of it. This is the empirical justification
for using the spatially resolved residual rather than a reconstruction-quality threshold.

## 7. Paired residuals, anatomy controlled

Comparing residuals for the same eye and clock position at both timepoints controls for anatomy:
any difference is attributable to lesion development rather than inter-animal variation. The
annotated mask is overlaid on the endpoint residual to locate the response. The forward pass is
written out again here rather than called from Section 5, so this section can be read on its own.

In [ ]:
# the forward pass is written out here rather than called from Section 5, so
# this section stands alone; the encoder draws a random tensor on every pass,
# so the seed is reset first
torch.manual_seed(SEED)
pairs = [('a baseline QC scan', 'an endpoint scan'),
         ('a baseline QC scan', 'an endpoint scan')]

fig, axes = plt.subplots(len(pairs), 4, figsize=(17, 3.5 * len(pairs)))
for row in range(len(pairs)):
    qc_name, ep_name = pairs[row]

    # QC first, then endpoint, one forward pass each
    sources = [(QC_FOLDER, qc_name), (ENDPOINT_FOLDER, ep_name)]
    displays = []
    residuals = []
    for folder, name in sources:
        gray = np.array(Image.open(os.path.join(folder, name)))[:, :, 0]
        resized = cv2.resize(gray, (768, 496))
        x = torch.from_numpy(resized).float().unsqueeze(0).unsqueeze(0).to(device) / 255.0
        with torch.no_grad():
            recon, _ = model(x)
        original = x.squeeze().cpu().numpy()
        reconstruction = recon.squeeze().cpu().numpy().T
        displays.append(resized)
        residuals.append(np.abs(original - reconstruction))

    qc_display = displays[0]
    ep_display = displays[1]
    qc_residual = residuals[0]
    ep_residual = residuals[1]

    mask = cv2.imread(os.path.join(MASK_FOLDER,
                                   ep_name.replace('.tif', '_mask.png')),
                      cv2.IMREAD_GRAYSCALE)
    mask = cv2.resize(mask, (768, 496), interpolation=cv2.INTER_NEAREST) > 127

    axes[row][0].imshow(qc_display, cmap='gray')
    axes[row][0].set_title(f"QC: {qc_name[:-4]}", fontsize=10)
    axes[row][1].imshow(qc_residual, cmap='hot', vmin=0, vmax=0.5)
    axes[row][1].set_title(f"QC residual (mean {qc_residual.mean():.4f})", fontsize=10)
    axes[row][2].imshow(ep_display, cmap='gray')
    axes[row][2].set_title(f"Endpoint: {ep_name[:-4]}", fontsize=10)
    axes[row][3].imshow(ep_residual, cmap='hot', vmin=0, vmax=0.5)
    axes[row][3].set_title(f"Endpoint residual (mean {ep_residual.mean():.4f})",
                           fontsize=10)

    # the annotated lesion outlined on the endpoint residual
    axes[row][3].contour(mask, levels=[0.5], colors='#39FF6A', linewidths=1.2)

    for col in range(4):
        axes[row][col].axis('off')

fig.suptitle('QC and endpoint residuals, same eye and clock position '
             '(green outline = annotated lesion)', y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(IMAGES_DIR, '02_qc_vs_endpoint.png'), bbox_inches='tight')
plt.show()

**How to read it.** Each row is one eye at one position, imaged twice, QC baseline first, endpoint later. Within a row the QC baseline scan, its error map, the endpoint scan, its error map. The **green
outline** on the last panel is where the expert marked the lesion.

Read the second row first, where error clearly increases inside the green outline. Then note that
**the two rows disagree**: on the top row the QC baseline scan actually has the higher average error.
That inconsistency is the finding, not a mistake.

Residual activity is present along the retinal band at both timepoints, since reconstruction is
imperfect everywhere. What changes at endpoint is the location of maximum response, which moves
inside the annotated contour. The residual is therefore lesion-specific rather than a general map
of reconstruction difficulty the quantitative form of this is measured in Section 8.

## 8. Residual discriminability

Mean intensity inside the annotated region against outside, computed on both the image and the
residual across all 74 annotated images, using boolean mask indexing. The ratio is the
lesion-to-background contrast; 1.0 would indicate no discrimination. Measuring both channels on
the same images makes the two directly comparable.

In [ ]:
# a contrast ratio of 1.0 would mean the channel cannot tell lesion from
# background at all; higher means a stronger separation
image_inside = []
image_outside = []
residual_inside = []
residual_outside = []

for name in anno_files:
    gray = np.array(Image.open(os.path.join(ENDPOINT_FOLDER, name)))[:, :, 0]
    gray = cv2.resize(gray, (768, 496)).astype(np.float32) / 255.0
    mask = cv2.imread(os.path.join(MASK_FOLDER, name.replace('.tif', '_mask.png')),
                      cv2.IMREAD_GRAYSCALE)
    mask = cv2.resize(mask, (768, 496), interpolation=cv2.INTER_NEAREST) > 127
    residual = residual_cache[name]

    image_inside.append(gray[mask].mean())
    image_outside.append(gray[~mask].mean())
    residual_inside.append(residual[mask].mean())
    residual_outside.append(residual[~mask].mean())

image_inside = np.array(image_inside)
image_outside = np.array(image_outside)
residual_inside = np.array(residual_inside)
residual_outside = np.array(residual_outside)

print(f"Measured lesion contrast on {len(anno_files)} images")

In [ ]:
image_ratio = image_inside.mean() / image_outside.mean()
residual_ratio = residual_inside.mean() / residual_outside.mean()
image_consistent = int((image_inside > image_outside).sum())
residual_consistent = int((residual_inside > residual_outside).sum())

print(f"{'Channel':<12} {'inside':>9} {'outside':>9} {'ratio':>8} {'consistent':>12}")
print("=" * 56)
print(f"{'Image':<12} {image_inside.mean():>9.4f} {image_outside.mean():>9.4f} "
      f"{image_ratio:>8.3f} {f'{image_consistent}/{len(anno_files)}':>12}")
print(f"{'Residual':<12} {residual_inside.mean():>9.4f} {residual_outside.mean():>9.4f} "
      f"{residual_ratio:>8.3f} {f'{residual_consistent}/{len(anno_files)}':>12}")
print(f"\nResidual contrast relative to image contrast: "
      f"{residual_ratio / image_ratio:.3f}x")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.0))

# dashed diagonal is 'no difference'; points above it mean the lesion stands
# out from its surroundings in that channel
axes[0].scatter(image_outside, image_inside, s=26, alpha=0.75, color=TRAIN,
                edgecolor='white')
limit = [0, max(image_inside.max(), image_outside.max()) * 1.05]
axes[0].plot(limit, limit, 'k--', linewidth=1, label='no difference')
axes[0].set_xlabel('mean outside lesion')
axes[0].set_ylabel('mean inside lesion')
axes[0].set_title(f"Image: ratio {image_ratio:.2f}")
axes[0].legend()

axes[1].scatter(residual_outside, residual_inside, s=26, alpha=0.75, color=VALID,
                edgecolor='white')
limit = [0, max(residual_inside.max(), residual_outside.max()) * 1.05]
axes[1].plot(limit, limit, 'k--', linewidth=1, label='no difference')
axes[1].set_xlabel('mean outside lesion')
axes[1].set_ylabel('mean inside lesion')
axes[1].set_title(f"Residual: ratio {residual_ratio:.2f}")
axes[1].legend()

bars = axes[2].bar(['Image', 'Residual'], [image_ratio, residual_ratio],
                   color=[TRAIN, VALID], alpha=0.9, edgecolor='white')
axes[2].axhline(1.0, color='0.4', linestyle='--', linewidth=1.2,
                label='no discrimination')
for bar, value in zip(bars, [image_ratio, residual_ratio]):
    axes[2].text(bar.get_x() + bar.get_width() / 2, value + 0.15,
                 f"{value:.3f}", ha='center', fontweight='bold')
axes[2].set_ylabel('lesion / background contrast')
axes[2].set_title(f"Residual is {residual_ratio / image_ratio:.3f}x the image")
axes[2].legend()

fig.suptitle('Lesion contrast in the image and in the residual, 74 images', y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(IMAGES_DIR, '02_residual_contrast.png'), bbox_inches='tight')
plt.show()

**How to read it.** The first two panels ask the same question of two different signals: does the
lesion stand out from its surroundings? Each dot is one image, with the lesion's brightness up and
the background's brightness across. The **dashed diagonal is where the two would be equal**, so how
far a dot sits above that line is how strongly the lesion stands out.

**Check the axis numbers before comparing the two panels** the left runs to about 0.9 and the
middle only to about 0.10, so the middle panel is far more zoomed in than it first appears. The
third panel exists to put both on one honest scale: 6.954 against 1.675.

Image contrast **6.954** (74/74 consistent); residual contrast **1.675** (69/74). Relative
discriminability **0.241x**. The residual is lesion-specific but a weaker discriminator than the
image it derives from.

Since `residual = |image − f(image)|` for deterministic `f`, it is a function of the input and
introduces no information not already present. It is a precomputed feature, not an additional
measurement, so any benefit is attributable to inductive bias rather than information content.
The prediction this sets up a change in error characteristics rather than in overlap is tested
in notebook 04.

This measurement withdrew an earlier claim of approximately six-fold amplification, which had
attributed the image's contrast (6.954) to the residual.

## 9. Consolidated reconstruction results

Values feeding `docs/NUMBERS.md` section 2 and the Reconstruction section of the paper.

In [ ]:
summary = [
    ('Fine-tuned on', f"{cfg['qc_train']} QC images, {cfg['qc_val']} held out"),
    ('Epochs run', f"{len(h['train_loss'])}, best at {ckpt['epoch']}, "
                   f"early stopped at {cfg['early_stopped_at']}"),
    ('Held-out SSIM before', f"{ckpt['baseline_ssim']:.4f}"),
    ('Held-out SSIM after', f"{ckpt['held_out_ssim']:.4f} ({improvement:+.4f})"),
    ('SSIM on QC images', f"{qc_scores.mean():.4f} (n={len(qc_scores)})"),
    ('SSIM on endpoint images', f"{ep_scores.mean():.4f} (n={len(ep_scores)})"),
    ('QC minus endpoint SSIM', f"{difference:+.4f}"),
    ('Residuals cached', f"{len(residual_cache)} at 496 x 768"),
    ('Image lesion contrast', f"{image_ratio:.3f}x ({image_consistent}/{len(anno_files)} images)"),
    ('Residual lesion contrast', f"{residual_ratio:.3f}x ({residual_consistent}/{len(anno_files)} images)"),
    ('Residual relative to image', f"{residual_ratio / image_ratio:.3f}x"),
]

print(f"{'Item':<30} Value")
print("=" * 74)
for name, value in summary:
    print(f"{name:<30} {value}")

In [ ]:
print("Figures saved in outputs/images/:")
for name in sorted(os.listdir(IMAGES_DIR)):
    if name.startswith('02_'):
        print(f"  {name}")

---

## Summary of notebook 02

Fine-tuning adapts OCT-F from **human** retina to **animal** retina using only the QC baseline scans. Held-out structural similarity on 19 images that never contributed a
weight update rises from **0.2427 to 0.6491**, an improvement of **+0.4063**, with a
train/validation gap of only **0.0218** at the final epoch.

Two findings temper what the residual can be expected to do.

**Whole-image SSIM is not a lesion detector.** QC baseline scans score 0.6490 and endpoint scans 0.6539 a difference of **−0.0050** in the wrong direction. A
single score over the whole frame cannot see a lesion covering 2% of it, so the
spatially resolved residual is what has to be used.

**The residual is real but weak.** It separates lesion from background on 69 of 74
images, but its contrast ratio is **1.675** against **6.954** for the image itself,
which is **0.241x** relative. Since it is `|image − f(image)|` for a deterministic `f`,
it carries no information the image does not already carry. It is a pre-computed
feature that may make an existing distinction easier to exploit a claim about
optimisation, not about information.